In [ ]:
import sys, os

## Include previous level directories to the module import path
sys.path.insert(0, os.path.abspath(os.path.join("..")))

# import arviz as az
import numpy as np
import h5py

# Export the results to GetDist

# Notice loadMCSamples requires a *full path*
import os
from matplotlib import pyplot as plt
import matplotlib

from matplotlib import cm

matplotlib.rcParams.update(
    {
        "font.size": 16,
        "axes.labelsize": 24,
        "axes.titlesize": 18,
        "text.usetex": True,
        "xtick.major.width": 1.5,
        "xtick.minor.width": 1.2,
        "ytick.major.width": 1.5,
        "ytick.minor.width": 1.2,
        "legend.fontsize": 18,
    }
)

# Deep purples / indigos
c_flatirons = "#440154"
c_flatirons_l = "#472d7b"
c_flatirons_ll = "#3b528b"

# Teal / cyan region
c_sunshine = "#2a788e"
c_sunshine_l = "#21908d"
c_sunshine_ll = "#20a386"

# Green-cyan (maximum brightness allowed)
c_skyline = "#35b779"
c_skyline_l = "#5ec962"  # Cap brightness here
# c_skyline_ll = omitted due to low print contrast

# Blue highlight
c_midnight = "#31688e"


cmap = cm.get_cmap("viridis")

# change fontsize
matplotlib.rcParams.update(
    {"font.size": 12, "axes.labelsize": 16, "axes.titlesize": 14}
)

# matplotlib.use('PDF')
save_figure = lambda filename: plt.savefig(
    "{}.pdf".format(filename), format="pdf", dpi=300
)

outfile = "../cddf_all/"


In [ ]:
sys.path.insert(0, os.path.abspath(os.path.join("../../DLA_data")))
import dla_data

In [ ]:

import numpy as np
from scipy.interpolate import PchipInterpolator

# From Table 2 (Results for Spline Model — Figure 7)
logN_nodes = np.array([12.0, 15.0, 17.0, 18.0, 
                       20.0, 21.0, 21.5, 22.0])
logf_nodes = np.array([-9.72, -14.41, -17.94, -19.39,
                       -21.28, -22.82, -23.95, -25.50])

# Paper specifies cubic Hermite spline (Fritsch & Carlson 1980)
cddf_spline = PchipInterpolator(logN_nodes, logf_nodes)

def log10_f_cddf(logN):
    """
    CDDF used in Figure 7 of Prochaska et al. 2014.
    A cubic Hermite spline defined by Table 2 (Spline Model).
    """
    logN = np.asarray(logN)
    logN_clip = np.clip(logN, logN_nodes[0], logN_nodes[-1])
    return cddf_spline(logN_clip)

def f_cddf(logN):
    return 10**log10_f_cddf(logN)


## GP-LLS

In [ ]:
def zbins_from_zmid_uniform(z_mid):
    z_mid = np.asarray(z_mid, dtype=float)
    dz = np.median(np.diff(z_mid))   # robust
    zbins = np.concatenate([
        [z_mid[0] - 0.5 * dz],
        z_mid + 0.5 * dz
    ])
    return zbins



def plot_line_density(
    z_cent: np.ndarray,
    dNdX: np.ndarray,
    dndx68: np.ndarray,
    dndx95: np.ndarray,
    zmin: float = 2.0,
    zmax: float = 5.5,
    label: str = "GP",
    color="blue",
    bins_per_z: int = 6,
    use_zcent_for_xerr: bool = True,
):
    """Plot the line density as a function of redshift"""
    if use_zcent_for_xerr:
        z_bins = zbins_from_zmid_uniform(z_cent)
    else:
        # Get the redshifts
        nbins = np.max([int((zmax - zmin) * bins_per_z), 1])
        z_bins = np.linspace(zmin, zmax, nbins + 1)

        # try match the zmax : sometimes not enough z bins to reach zmax
        # cut off the extra part until z_bins just larger than last z_cent
        # remove the last bin edge
        ind = z_bins > z_cent[-1]
        z_bins = z_bins[: np.where(ind)[0][0] + 1]

    try:
        # hope this line works
        xerrs = (z_cent - z_bins[:-1], z_bins[1:] - z_cent)
    except ValueError as e:
        xerrs = (z_cent - z_bins[:-2], z_bins[1:-1] - z_cent)

    # 2 sigma contours.
    plt.fill_between(z_cent, dndx95[:, 0], dndx95[:, 1], color="grey", alpha=0.5)
    yerr = (dNdX - dndx68[:, 0], dndx68[:, 1] - dNdX)
    plt.errorbar(z_cent, dNdX, yerr=yerr, xerr=xerrs, fmt="o", color=color, label=label)
    plt.xlabel(r"z")
    plt.ylabel(r"dN/dX")
    plt.xlim(zmin, zmax)

In [ ]:

# Load DLA data Noterdaeme 2012, Prochaska 2005
dla_data.dndx_not()
dla_data.dndx_pro()

subdir = "/Users/jibanmac/Documents/GitHub/desi_gpy_dla_detection_public/cddf_all/dla_cddf_20260107/desi_snr6.0_zqsos_2.15-5.0_dla_5.5_occam_1.0_dla_model_1_zminlyb_22.0_minobswave_3700.0_bins_5_lnhi_17.2-22.0_lnhi_dndx_17.2-19.0/"

# dN/dX: Fiducial
dndx_all = np.loadtxt(os.path.join(subdir, "dndx_all.txt"))
(_, N) = dndx_all.shape

dndx68 = np.full((N, 2), fill_value=np.nan)
dndx95 = np.full((N, 2), fill_value=np.nan)
(
    z_cent,
    dNdX,
    dndx68[:, 0],
    dndx68[:, 1],
    dndx95[:, 0],
    dndx95[:, 1],
) = dndx_all

plot_line_density(z_cent, dNdX, dndx68, dndx95, color="blue", label="GP-LLS (SNR > 6)", bins_per_z=5, zmin=2, zmax=4.25)

# dN/dX: High SNR
high_snrsubdir = "/Users/jibanmac/Documents/GitHub/desi_gpy_dla_detection_public/cddf_all/dla_cddf_20260107/desi_snr8.0_zqsos_2.15-5.0_dla_5.5_occam_1.0_dla_model_1_zminlyb_22.0_minobswave_3700.0_bins_5_lnhi_17.2-22.0_lnhi_dndx_17.2-19.0/"
dndx_all = np.loadtxt(os.path.join(high_snrsubdir, "dndx_all.txt"))
(_, N) = dndx_all.shape

dndx68 = np.full((N, 2), fill_value=np.nan)
dndx95 = np.full((N, 2), fill_value=np.nan)
(
    z_cent,
    dNdX,
    dndx68[:, 0],
    dndx68[:, 1],
    dndx95[:, 0],
    dndx95[:, 1],
) = dndx_all

plot_line_density(z_cent, dNdX, dndx68, dndx95, color="C0", label="GP-LLS (SNR > 8)", bins_per_z=5, zmin=2, zmax=4.25)

plt.xlim(1.95, 5.3)
plt.legend()

save_figure(os.path.join(outfile, "dndx_lls"))

## Truth mock

In [ ]:
from astropy.table import Table

dla_cat = Table.read(
    "/Users/jibanmac/Documents/GitHub/desi_gpy_dla_detection_public/data/london/dla_cat.fits"
)
qso_cat = Table.read(
    "/Users/jibanmac/Documents/GitHub/desi_gpy_dla_detection_public/data/london/zcat.fits"
)

In [ ]:
# ============================================================
# DLA/LLS/subDLA summary statistics: dN/dX and CDDF
#
# Correct CDDF convention (Bird+ 2016 / arXiv:1610.01165):
#   f(N) = d^2 N_abs / (dN dX)
# where N is LINEAR column density in cm^-2, and X is dimensionless
#
# This file computes:
#   - dN/dX in redshift bins (for any logNHI range)
#   - CDDF f(N) in (z-bin, logN-bin) grids, where the binning is in log10 N
#     but normalization divides by ΔN = 10^{logN_hi} - 10^{logN_lo}.
#
# Assumptions:
#   - QSO catalog contains TARGETID, Z (QSO redshift)
#   - absorber catalog contains TARGETID, Z_DLA, NHI
#   - NHI column is log10(NHI/cm^2) if assume_logNHI=True, else linear NHI
# ============================================================

import numpy as np

# ----------------------------
# Constants
# ----------------------------
C_KMS = 299792.458

# ----------------------------
# 1) Cosmology / path length
# ----------------------------

def HubbleByH0(z, Omega_m=0.279):
    """
    H(z)/H0 for flat LCDM with Omega_m, Omega_L = 1 - Omega_m
    """
    z = np.asarray(z, dtype=float)
    return np.sqrt(Omega_m * (1.0 + z) ** 3 + (1.0 - Omega_m))


def path_length_int(z, Omega_m=0.279):
    """
    Your exact integrand:
      dX/dz = (1+z)^2 / (H(z)/H0)
    """
    z = np.asarray(z, dtype=float)
    return (1.0 + z) ** 2 / HubbleByH0(z, Omega_m)


class AbsorptionDistance:
    """
    Fast helper to compute X(z)=∫ dX/dz dz and ΔX via grid + interpolation.
    Uses your exact dX/dz definition.
    """
    def __init__(self, zmax, Omega_m=0.279, ngrid=40001):
        self.Omega_m = float(Omega_m)
        self.zgrid = np.linspace(0.0, float(zmax), int(ngrid))
        integrand = path_length_int(self.zgrid, Omega_m=self.Omega_m)

        dz = np.diff(self.zgrid)
        X = np.empty_like(self.zgrid)
        X[0] = 0.0
        X[1:] = np.cumsum(0.5 * (integrand[:-1] + integrand[1:]) * dz)
        self.Xgrid = X

    def X(self, z):
        z = np.asarray(z, dtype=float)
        return np.interp(z, self.zgrid, self.Xgrid)

    def deltaX(self, z1, z2):
        z1 = np.asarray(z1, dtype=float)
        z2 = np.asarray(z2, dtype=float)
        return self.X(z2) - self.X(z1)


# ----------------------------
# 2) QSO searchable windows
# ----------------------------

def zmax_nonprox(z_qso, v_prox_kms=10000.0):
    """
    Proximate cut:
      z_max = z_qso - (1+z_qso) * v/c
    """
    z_qso = np.asarray(z_qso, dtype=float)
    return z_qso - (1.0 + z_qso) * (v_prox_kms / C_KMS)


def build_qso_windows(qso_cat, *, zmin, zmax_global=None, v_prox_kms=10000.0):
    """
    Build per-QSO absorber windows [z_lo, z_hi].

    qso_cat must provide columns:
      - TARGETID
      - Z   (QSO redshift)

    zmin is a global floor; if you have per-QSO wavelength coverage, replace this.
    """
    tid = np.asarray(qso_cat["TARGETID"])
    zq = np.asarray(qso_cat["Z"], dtype=float)

    z_lo = np.full_like(zq, float(zmin), dtype=float)
    z_hi = zmax_nonprox(zq, v_prox_kms=v_prox_kms)

    if zmax_global is not None:
        z_hi = np.minimum(z_hi, float(zmax_global))

    ok = np.isfinite(z_lo) & np.isfinite(z_hi) & (z_hi > z_lo)
    return tid[ok], z_lo[ok], z_hi[ok]


# ----------------------------
# 3) Filter absorbers to selected QSO sample + windows
# ----------------------------

def filter_absorbers_to_qsos(
    abs_cat,
    qso_tid, qso_zlo, qso_zhi,
    *,
    logNHImin=None,
    logNHImax=None,
    assume_logNHI=True,
):
    """
    Keep only absorbers that:
      - have TARGETID in the provided QSO catalog
      - satisfy z_lo <= z_abs <= z_hi for that QSO
      - satisfy logNHI cuts (if provided)

    abs_cat must provide:
      - TARGETID
      - Z_DLA (absorber redshift)
      - NHI (log10NHI if assume_logNHI=True else linear NHI)
    """
    tid_abs = np.asarray(abs_cat["TARGETID"])
    z_abs = np.asarray(abs_cat["Z_DLA"], dtype=float)
    NHIcol = np.asarray(abs_cat["NHI"], dtype=float)

    logN = NHIcol if assume_logNHI else np.log10(NHIcol)

    # map TARGETID -> QSO index using sorted search
    sort_idx = np.argsort(qso_tid)
    qso_tid_sorted = qso_tid[sort_idx]

    pos = np.searchsorted(qso_tid_sorted, tid_abs)
    in_bounds = (pos >= 0) & (pos < len(qso_tid_sorted))
    match = in_bounds & (qso_tid_sorted[pos.clip(0, len(qso_tid_sorted) - 1)] == tid_abs)

    qso_idx = sort_idx[pos[match]]
    z_m = z_abs[match]
    logN_m = logN[match]
    tid_m = tid_abs[match]

    # window cut
    zlo = qso_zlo[qso_idx]
    zhi = qso_zhi[qso_idx]
    w = (z_m >= zlo) & (z_m <= zhi)

    z_w = z_m[w]
    logN_w = logN_m[w]
    tid_w = tid_m[w]
    qso_idx_w = qso_idx[w]

    # column density cuts
    if logNHImin is not None:
        m = logN_w >= float(logNHImin)
        z_w, logN_w, tid_w, qso_idx_w = z_w[m], logN_w[m], tid_w[m], qso_idx_w[m]
    if logNHImax is not None:
        m = logN_w <= float(logNHImax)
        z_w, logN_w, tid_w, qso_idx_w = z_w[m], logN_w[m], tid_w[m], qso_idx_w[m]

    return z_w, logN_w, tid_w, qso_idx_w


# ----------------------------
# 4) Total ΔX in z bins
# ----------------------------

def total_DeltaX_in_zbins(zbins, qso_zlo, qso_zhi, Xcalc):
    """
    ΔX_k = sum_i ∫_{W_i ∩ [z_k,z_{k+1}]} dX
    """
    zbins = np.asarray(zbins, dtype=float)
    nb = len(zbins) - 1
    X_tot = np.zeros(nb, dtype=float)

    for k in range(nb):
        lo, hi = zbins[k], zbins[k + 1]
        o_lo = np.maximum(qso_zlo, lo)
        o_hi = np.minimum(qso_zhi, hi)
        m = o_hi > o_lo
        if np.any(m):
            X_tot[k] = np.sum(Xcalc.deltaX(o_lo[m], o_hi[m]))
    return X_tot


# ----------------------------
# 5) dN/dX
# ----------------------------

def compute_dndx(
    abs_cat, qso_cat,
    *,
    zbins,
    zmin,
    zmax_global=None,
    v_prox_kms=10000.0,
    Omega_m=0.279,
    logNHImin=20.3,
    logNHImax=23.0,
    assume_logNHI=True,
    n_boot=0,
    rng=None,
):
    """
    dN/dX in z-bins for absorbers with logNHI in [logNHImin, logNHImax].
    abs_cat can be DLA/LLS/subDLA catalog; only the logNHI range matters.
    """
    zbins = np.asarray(zbins, dtype=float)
    z_mid = 0.5 * (zbins[:-1] + zbins[1:])

    # QSO windows
    qso_tid, qso_zlo, qso_zhi = build_qso_windows(
        qso_cat, zmin=zmin, zmax_global=zmax_global, v_prox_kms=v_prox_kms
    )

    if len(qso_tid) == 0:
        raise ValueError("No QSOs left after applying zmin/prox cuts; cannot compute dN/dX.")

    # Absorption distance
    Xcalc = AbsorptionDistance(zmax=float(np.max(qso_zhi)), Omega_m=Omega_m)

    # Total ΔX per z-bin
    X_tot = total_DeltaX_in_zbins(zbins, qso_zlo, qso_zhi, Xcalc)

    # Filter absorbers to selected QSOs + windows + logNHI cuts
    z_abs, logN, tid_abs, qso_idx_abs = filter_absorbers_to_qsos(
        abs_cat, qso_tid, qso_zlo, qso_zhi,
        logNHImin=logNHImin, logNHImax=logNHImax,
        assume_logNHI=assume_logNHI,
    )

    # Counts per z-bin
    N_abs, _ = np.histogram(z_abs, bins=zbins)

    dndx = np.where(X_tot > 0, N_abs / X_tot, np.nan)
    err_pois = np.where(X_tot > 0, np.sqrt(N_abs) / X_tot, np.nan)

    # Bootstrap over QSOs (sightlines)
    err_boot = None
    if n_boot and n_boot > 0:
        rng = np.random.default_rng() if rng is None else rng
        nq = len(qso_tid)
        nb = len(z_mid)

        zbin = np.digitize(z_abs, zbins) - 1
        valid = (zbin >= 0) & (zbin < nb)
        zbin = zbin[valid]
        qso_idx_abs = qso_idx_abs[valid]

        # per-QSO counts in each zbin
        per_qso_counts = np.zeros((nq, nb), dtype=int)
        np.add.at(per_qso_counts, (qso_idx_abs, zbin), 1)

        # per-QSO ΔX contributions in each zbin
        per_qso_X = np.zeros((nq, nb), dtype=float)
        for k in range(nb):
            lo, hi = zbins[k], zbins[k + 1]
            o_lo = np.maximum(qso_zlo, lo)
            o_hi = np.minimum(qso_zhi, hi)
            m = o_hi > o_lo
            if np.any(m):
                per_qso_X[m, k] = Xcalc.deltaX(o_lo[m], o_hi[m])

        boot = np.empty((n_boot, nb), dtype=float)
        for b in range(n_boot):
            draw = rng.integers(0, nq, size=nq)
            Nb = per_qso_counts[draw].sum(axis=0)
            Xb = per_qso_X[draw].sum(axis=0)
            boot[b] = np.where(Xb > 0, Nb / Xb, np.nan)

        err_boot = np.nanstd(boot, axis=0, ddof=1)

    return {
        "z_mid": z_mid,
        "zbins": zbins,
        "dndx": dndx,
        "err_poisson": err_pois,
        "err_boot": err_boot,
        "N_abs": N_abs,
        "X_tot": X_tot,
        "meta": {
            "logNHImin": float(logNHImin),
            "logNHImax": float(logNHImax),
            "Omega_m": float(Omega_m),
            "v_prox_kms": float(v_prox_kms),
        },
    }


# ----------------------------
# 6) CDDF f(N): d^2N_abs / (dN dX)   [correct paper convention]
# ----------------------------

def compute_cddf_fN(
    abs_cat, qso_cat,
    *,
    zbins,
    logN_bins,
    zmin,
    zmax_global=None,
    v_prox_kms=10000.0,
    Omega_m=0.279,
    logNHImin=None,
    logNHImax=None,
    assume_logNHI=True,
    n_boot=0,
    rng=None,
):
    """
    Compute CDDF in Bird+ convention:
        f(N) = d^2 N_abs / (dN dX)
    using log10N bins but dividing by ΔN (linear).

    Returns arrays:
      - z_mid (nbz,)
      - logN_mid (nbn,)
      - N_mid (nbn,) linear column density at bin center (geometric mean)
      - fN (nbz, nbn)
      - err_poisson (nbz, nbn)
      - err_boot (nbz, nbn) or None
    """
    zbins = np.asarray(zbins, dtype=float)
    logN_bins = np.asarray(logN_bins, dtype=float)

    z_mid = 0.5 * (zbins[:-1] + zbins[1:])
    nbz = len(z_mid)
    nbn = len(logN_bins) - 1

    # Linear bin edges and widths ΔN (cm^-2)
    N_edges = 10.0 ** logN_bins
    dN = np.diff(N_edges)  # (nbn,)

    # Geometric-mean bin center (standard for log bins)
    N_mid = np.sqrt(N_edges[:-1] * N_edges[1:])
    logN_mid = np.log10(N_mid)

    # QSO windows
    qso_tid, qso_zlo, qso_zhi = build_qso_windows(
        qso_cat, zmin=zmin, zmax_global=zmax_global, v_prox_kms=v_prox_kms
    )

    if len(qso_tid) == 0:
        raise ValueError("No QSOs left after applying zmin/prox cuts; cannot compute CDDF.")

    # Absorption distance and ΔX per z-bin
    Xcalc = AbsorptionDistance(zmax=float(np.max(qso_zhi)), Omega_m=Omega_m)
    X_tot = total_DeltaX_in_zbins(zbins, qso_zlo, qso_zhi, Xcalc)

    # Filter absorbers to selected QSOs + windows (+ optional logN truncation)
    z_abs, logN, tid_abs, qso_idx_abs = filter_absorbers_to_qsos(
        abs_cat, qso_tid, qso_zlo, qso_zhi,
        logNHImin=logNHImin, logNHImax=logNHImax,
        assume_logNHI=assume_logNHI,
    )

    # 2D binning
    zbin = np.digitize(z_abs, zbins) - 1
    nbin = np.digitize(logN, logN_bins) - 1
    valid = (zbin >= 0) & (zbin < nbz) & (nbin >= 0) & (nbin < nbn)

    zbin = zbin[valid]
    nbin = nbin[valid]
    qso_idx_abs = qso_idx_abs[valid]

    counts = np.zeros((nbz, nbn), dtype=int)
    np.add.at(counts, (zbin, nbin), 1)

    # f(N) normalization: counts / (ΔX * ΔN)
    fN = np.full_like(counts, np.nan, dtype=float)
    err_pois = np.full_like(counts, np.nan, dtype=float)
    for k in range(nbz):
        if X_tot[k] > 0:
            fN[k] = counts[k] / (X_tot[k] * dN)
            err_pois[k] = np.sqrt(counts[k]) / (X_tot[k] * dN)

    # Bootstrap over QSOs (sightlines)
    err_boot = None
    if n_boot and n_boot > 0:
        rng = np.random.default_rng() if rng is None else rng
        nq = len(qso_tid)

        # per-QSO ΔX contributions in each zbin
        per_qso_X = np.zeros((nq, nbz), dtype=float)
        for k in range(nbz):
            lo, hi = zbins[k], zbins[k + 1]
            o_lo = np.maximum(qso_zlo, lo)
            o_hi = np.minimum(qso_zhi, hi)
            m = o_hi > o_lo
            if np.any(m):
                per_qso_X[m, k] = Xcalc.deltaX(o_lo[m], o_hi[m])

        # per-QSO 2D counts in (zbin, nbin)
        per_qso_counts = np.zeros((nq, nbz, nbn), dtype=int)
        np.add.at(per_qso_counts, (qso_idx_abs, zbin, nbin), 1)

        boot = np.empty((n_boot, nbz, nbn), dtype=float)
        for b in range(n_boot):
            draw = rng.integers(0, nq, size=nq)
            Cb = per_qso_counts[draw].sum(axis=0)  # (nbz, nbn)
            Xb = per_qso_X[draw].sum(axis=0)       # (nbz,)

            fb = np.full((nbz, nbn), np.nan, dtype=float)
            for k in range(nbz):
                if Xb[k] > 0:
                    fb[k] = Cb[k] / (Xb[k] * dN)
            boot[b] = fb

        err_boot = np.nanstd(boot, axis=0, ddof=1)

    return {
        "z_mid": z_mid,
        "zbins": zbins,
        "logN_mid": logN_mid,
        "logN_bins": logN_bins,
        "N_mid": N_mid,
        "N_edges": N_edges,
        "dN": dN,
        "fN": fN,
        "err_poisson": err_pois,
        "err_boot": err_boot,
        "counts": counts,
        "X_tot": X_tot,
        "meta": {
            "Omega_m": float(Omega_m),
            "v_prox_kms": float(v_prox_kms),
            "logNHImin": None if logNHImin is None else float(logNHImin),
            "logNHImax": None if logNHImax is None else float(logNHImax),
        },
    }


# ----------------------------
# 7) Optional: plotting utilities (matplotlib)
# ----------------------------

def plot_dndx(out, *, label=None, prefer_boot=True, ax=None, show=True):
    import matplotlib.pyplot as plt

    z = out["z_mid"]
    y = out["dndx"]
    yerr = out["err_boot"] if (prefer_boot and out.get("err_boot") is not None) else out["err_poisson"]

    # xerr from zbins
    zb = out["zbins"]
    xerr = 0.5 * (zb[1:] - zb[:-1])

    if ax is None:
        fig, ax = plt.subplots()

    ax.errorbar(z, y, yerr=yerr, xerr=xerr, fmt="o", capsize=2, label=label)
    ax.set_xlabel("z")
    ax.set_ylabel(r"$dN/dX$")
    ax.grid(True, alpha=0.3)
    if label:
        ax.legend()
    if show:
        plt.show()
    return ax


def plot_cddf_slice_fN(out, zbin_index, *, label=None, prefer_boot=True, ax=None, show=True, ylog=True):
    """
    Plot f(N) vs log10 N at a chosen z-bin index.
    """
    import matplotlib.pyplot as plt

    x = out["logN_mid"]
    y = out["fN"][zbin_index]
    yerr = out["err_boot"][zbin_index] if (prefer_boot and out.get("err_boot") is not None) else out["err_poisson"][zbin_index]

    if ax is None:
        fig, ax = plt.subplots()

    ax.errorbar(x, y, yerr=yerr, fmt="o", capsize=2, label=label)
    ax.set_xlabel(r"$\log_{10} N_{\rm HI}$")
    ax.set_ylabel(r"$f(N)=d^2N/(dN\,dX)$  [cm$^2$]")
    ax.grid(True, alpha=0.3)
    if ylog:
        ax.set_yscale("log")
    if label:
        ax.legend()
    if show:
        plt.show()
    return ax

In [ ]:
import numpy as np

# --- choose redshift bins ---
# zbins = np.array([2.0, 2.5, 3.0, 3.5, 4.0])
zbins = zbins_from_zmid_uniform(z_cent)

# --- common analysis settings ---
common = dict(
    zbins=zbins,
    zmin=2.0,
    zmax_global=None,
    v_prox_kms=3000.0,
    Omega_m=0.279,
    assume_logNHI=True,
    n_boot=200,   # set 0 to skip bootstrap
)

# --- LLS dN/dX ---
out_lls = compute_dndx(
    dla_cat, qso_cat,
    logNHImin=17.2, logNHImax=19.0,
    **common
)

# --- subDLA dN/dX ---
out_subdla = compute_dndx(
    dla_cat, qso_cat,
    logNHImin=19.0, logNHImax=20.3,
    **common
)

# --- DLA dN/dX ---
out_dla = compute_dndx(
    dla_cat, qso_cat,
    logNHImin=20.3, logNHImax=23.0,
    **common
)

print("z_mid:", out_dla["z_mid"])
print("DLA dN/dX:", out_dla["dndx"], "Poisson:", out_dla["err_poisson"], "Boot:", out_dla["err_boot"])

In [ ]:
logN_bins_lls = np.arange(17.2, 22 + 1e-6, 0.2)

cddf_lls = compute_cddf_fN(
    dla_cat, qso_cat,
    zbins=zbins,
    logN_bins=logN_bins_lls,
    zmin=2.0,
    v_prox_kms=3000.0,
    Omega_m=0.279,
    logNHImin=17.2, logNHImax=22,  # truncate to LLS
    assume_logNHI=True,
    n_boot=200,
)

# f has shape (n_zbins, n_logN_bins-1)
# print("LLS CDDF f shape:", cddf_lls["f"].shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Helpers
# ----------------------------

def _pick_err(err_poisson, err_boot=None, prefer_boot=True):
    """Return the error array to plot."""
    if prefer_boot and (err_boot is not None):
        return err_boot
    return err_poisson


# ----------------------------
# 7) Optional: plotting utilities (matplotlib)
# ----------------------------

def plot_dndx(out, *, label=None, prefer_boot=True, ax=None, show=True):
    import matplotlib.pyplot as plt

    z = out["z_mid"]
    y = out["dndx"]
    yerr = out["err_boot"] if (prefer_boot and out.get("err_boot") is not None) else out["err_poisson"]

    # xerr from zbins
    zb = out["zbins"]
    xerr = 0.5 * (zb[1:] - zb[:-1])

    if ax is None:
        fig, ax = plt.subplots()

    ax.errorbar(z, y, yerr=yerr, xerr=xerr, fmt="o", capsize=2, label=label)
    ax.set_xlabel("z")
    ax.set_ylabel(r"$dN/dX$")
    ax.grid(True, alpha=0.3)
    if label:
        ax.legend()
    if show:
        plt.show()
    return ax


def plot_cddf_slice_fN(out, zbin_index, *, label=None, prefer_boot=True, ax=None, show=True, ylog=True):
    """
    Plot f(N) vs log10 N at a chosen z-bin index.
    """
    import matplotlib.pyplot as plt

    x = out["logN_mid"]
    y = out["fN"][zbin_index]
    yerr = out["err_boot"][zbin_index] if (prefer_boot and out.get("err_boot") is not None) else out["err_poisson"][zbin_index]

    if ax is None:
        fig, ax = plt.subplots()

    ax.errorbar(x, y, yerr=yerr, fmt="o", capsize=2, label=label)
    ax.set_xlabel(r"$\log_{10} N_{\rm HI}$")
    ax.set_ylabel(r"$f(N)=d^2N/(dN\,dX)$  [cm$^2$]")
    ax.grid(True, alpha=0.3)
    if ylog:
        ax.set_yscale("log")
    if label:
        ax.legend()
    if show:
        plt.show()
    return ax

In [ ]:
# ------------------------------------------------------------
# 2) Plot dN/dX (overlay)
# ------------------------------------------------------------
fig, ax = plt.subplots()
plot_dndx(out_lls, label="LLS (17.2–19.0)", prefer_boot=True, ax=ax, show=False)
plot_dndx(out_subdla, label="subDLA (19.0–20.3)", prefer_boot=True, ax=ax, show=False)
plot_dndx(out_dla, label="DLA (20.3–23.0)", prefer_boot=True, ax=ax, show=False)
ax.set_title("Line density: dN/dX")


In [ ]:
# ------------------------------------------------------------
# 4) Plot one redshift slice of CDDF (overlay)
# Choose z-bin index k (0..len(zbins)-2)
# Example: k=2 corresponds to zbins[2]..zbins[3] = 3.0..3.5
# ------------------------------------------------------------
k = 2

fig, ax = plt.subplots()
plot_cddf_slice_fN(
    cddf_lls, k,
    label=f"z~{cddf_lls['z_mid'][k]:.2f}",
    prefer_boot=True, ax=ax, show=False, ylog=True
)
# plot_cddf_slice_fN(
#     cddf_sub, k,
#     label=f"subDLA z~{cddf_sub['z_mid'][k]:.2f}",
#     prefer_boot=True, ax=ax, show=False, ylog=True
# )
# plot_cddf_slice_fN(
#     cddf_dla, k,
#     label=f"DLA  z~{cddf_dla['z_mid'][k]:.2f}",
#     prefer_boot=True, ax=ax, show=False, ylog=True
# )

# plot the CDDF from Prochaska et al. 2014
logN_plot = np.linspace(17.2, 22.0, 500)
plt.plot(logN_plot, f_cddf(logN_plot), label="Prochaska et al. 2014 CDDF")
plt.yscale("log")
plt.xlabel(r"$\log_{10} N_{\mathrm{HI}}$ [cm$^{-2}$]")
plt.ylabel(r"$f(N_{\mathrm{HI}})$ [cm$^{2}$]")
plt.title("Column Density Distribution Function (CDDF)")

ax.set_title(f"CDDF slice in z-bin [{zbins[k]}, {zbins[k+1]}]")
plt.show()


## Compare with mock measurement

In [ ]:
# London mock
subdir = "/Users/jibanmac/Documents/GitHub/desi_gpy_dla_detection_public/cddf_all/dla_london_cddf_nhi172_20260107/desi_snr6.0_zqsos_2.15-5.0_dla_5.5_occam_1.0_dla_model_1_zminlyb_22.0_minobswave_3700.0_bins_5_lnhi_17.2-22.0_lnhi_dndx_17.2-19.0_z_dla_cddf_min_1.0_z_dla_dndx_min_2.0/"

# dN/dX: Fiducial
dndx_all = np.loadtxt(os.path.join(subdir, "dndx_all.txt"))
(_, N) = dndx_all.shape

dndx68_mock = np.full((N, 2), fill_value=np.nan)
dndx95_mock = np.full((N, 2), fill_value=np.nan)
(
    z_cent_mock,
    dNdX_mock,
    dndx68_mock[:, 0],
    dndx68_mock[:, 1],
    dndx95_mock[:, 0],
    dndx95_mock[:, 1],
) = dndx_all

######### SNR > 8 #########
# London mock
subdir = "/Users/jibanmac/Documents/GitHub/desi_gpy_dla_detection_public/cddf_all/dla_london_cddf_nhi172_20260107/desi_snr8.0_zqsos_2.15-5.0_dla_5.5_occam_1.0_dla_model_1_zminlyb_22.0_minobswave_3700.0_bins_5_lnhi_17.2-22.0_lnhi_dndx_17.2-19.0_z_dla_cddf_min_1.0_z_dla_dndx_min_2.0/"

# dN/dX: Fiducial
dndx_all = np.loadtxt(os.path.join(subdir, "dndx_all.txt"))
(_, N) = dndx_all.shape

dndx68_mock_highsnr = np.full((N, 2), fill_value=np.nan)
dndx95_mock_highsnr = np.full((N, 2), fill_value=np.nan)
(
    z_cent_mock_highsnr,
    dNdX_mock_highsnr,
    dndx68_mock_highsnr[:, 0],
    dndx68_mock_highsnr[:, 1],
    dndx95_mock_highsnr[:, 0],
    dndx95_mock_highsnr[:, 1],
) = dndx_all



fig, ax = plt.subplots()

plot_line_density(z_cent_mock, dNdX_mock, dndx68_mock, dndx95_mock, color=c_midnight, label="GP-LLS (SNR $>$ 6)", bins_per_z=5, zmin=2, zmax=4)
plot_line_density(z_cent_mock_highsnr, dNdX_mock_highsnr, dndx68_mock_highsnr, dndx95_mock_highsnr, color=c_flatirons, label="GP-LLS (SNR $>$ 8)", bins_per_z=5, zmin=2, zmax=4)

plot_dndx(out_lls, label="LLS (17.2-19.0)", prefer_boot=True, ax=ax, show=False)

Calibration factor

In [ ]:
import numpy as np

def sym_err_from_bounds(y, bounds68):
    """
    Convert absolute 68% bounds into a symmetric ~1σ uncertainty around y.

    Inputs
    ------
    y : array-like, shape (N,)
        Central estimate (e.g., median/mean) corresponding to the bounds.
    bounds68 : array-like, shape (N, 2)
        Absolute lower/upper bounds: [low, high], NOT error bars.

    Returns
    -------
    sigma : ndarray, shape (N,)
        Symmetric sigma ~ average of (y-low) and (high-y).
    """
    y = np.asarray(y, float)
    bounds68 = np.asarray(bounds68, float)
    low = bounds68[:, 0]
    high = bounds68[:, 1]
    return 0.5 * ((y - low) + (high - y))


def eval_truth_at_z(z_eval, out_truth, which_err="boot"):
    """
    Interpolate truth dN/dX (and its error) onto z_eval.
    """
    zt = np.asarray(out_truth["z_mid"], float)
    yt = np.asarray(out_truth["dndx"], float)

    if which_err == "boot" and out_truth.get("err_boot") is not None:
        et = np.asarray(out_truth["err_boot"], float)
    else:
        et = np.asarray(out_truth["err_poisson"], float)

    yti = np.interp(z_eval, zt, yt)
    eti = np.interp(z_eval, zt, et)
    return yti, eti


def calibration_factor_alpha(
    z_meas, y_meas, bounds68_meas,
    out_truth, *,
    truth_err_kind="boot",
    clip=None,
):
    """
    alpha(z) = y_true(z) / y_meas(z)

    bounds68_meas must be absolute bounds [low, high] (not +/- errors).
    We convert bounds -> symmetric sigma for propagation.

    Error propagation (independent):
      (σ_alpha/alpha)^2 = (σ_true/y_true)^2 + (σ_meas/y_meas)^2
    """
    z_meas = np.asarray(z_meas, float)
    y_meas = np.asarray(y_meas, float)
    bounds68_meas = np.asarray(bounds68_meas, float)

    # Convert bounds to a symmetric ~1σ
    yerr_meas = sym_err_from_bounds(y_meas, bounds68_meas)

    y_true, yerr_true = eval_truth_at_z(z_meas, out_truth, which_err=truth_err_kind)

    eps = 1e-30
    A = np.maximum(y_true, eps)
    B = np.maximum(y_meas, eps)

    alpha = A / B

    frac = np.sqrt((yerr_true / A) ** 2 + (yerr_meas / B) ** 2)
    alpha_err = alpha * frac

    if clip is not None:
        lo, hi = clip
        alpha = np.clip(alpha, lo, hi)

    # corrected curve (by construction should align with truth in expectation)
    y_corr = y_meas * alpha
    y_corr_err = y_corr * frac  # full propagated

    return {
        "z": z_meas,
        "y_meas": y_meas,
        "bounds68_meas": bounds68_meas,
        "yerr_meas_sym": yerr_meas,
        "y_true": y_true,
        "yerr_true": yerr_true,
        "alpha": alpha,
        "alpha_err": alpha_err,
        "y_corr": y_corr,
        "y_corr_err": y_corr_err,
    }

In [ ]:
# ------------------------------------------------------------
# Correct: dndx68_mock is [low, high] bounds, NOT error bars
# ------------------------------------------------------------

# measured ~1σ (symmetric) from absolute 68% bounds
err_meas = sym_err_from_bounds(dNdX_mock, dndx68_mock)

# calibration factor alpha(z) = truth / measured
cal = calibration_factor_alpha(
    z_meas=z_cent_mock,
    y_meas=dNdX_mock,
    bounds68_meas=dndx68_mock,   # <-- bounds [low, high]
    out_truth=out_lls,           # truth from catalog
    truth_err_kind="boot",       # or "poisson"
    clip=None
)

######### SNR > 8 #########
err_meas_highsnr = sym_err_from_bounds(dNdX_mock_highsnr, dndx68_mock_highsnr)

cal_highsnr = calibration_factor_alpha(
    z_meas=z_cent_mock_highsnr,
    y_meas=dNdX_mock_highsnr,
    bounds68_meas=dndx68_mock_highsnr,   # <-- bounds [low, high]
    out_truth=out_lls,           # truth from catalog
    truth_err_kind="boot",       # or "poisson"
    clip=None
)

# If you specifically still want err_meas as an array for quick inspection:
# err_meas = cal["yerr_meas_sym"]

In [ ]:
import matplotlib.pyplot as plt

# --- (A) overlay curves ---
fig, ax = plt.subplots()

# your measured curve (blue) — you already have plot_line_density
plot_line_density(
    z_cent_mock, dNdX_mock, dndx68_mock, dndx95_mock,
    color=c_midnight, label="Field-level measured dN/dX"
)
# SNR > 8
plot_line_density(
    z_cent_mock_highsnr, dNdX_mock_highsnr, dndx68_mock_highsnr, dndx95_mock_highsnr,
    color=c_flatirons, label="Field-level measured dN/dX (SNR $>$ 8)"
)

# truth (from catalog)
plot_dndx(out_lls, label="Truth (from catalog)", prefer_boot=True, ax=ax, show=False)

# (3) Calibrated curve (measured × alpha)
# This has a *derived* symmetric uncertainty, so errorbar is appropriate
ax.errorbar(
    cal["z"],
    cal["y_corr"],
    yerr=cal["y_corr_err"],
    fmt="o",
    color="black",
    capsize=2,
    label=r"Measured $\times \alpha(z)$",
)

ax.set_title("dN/dX: measured vs truth vs calibrated")
ax.legend()
plt.show()


# --- (B) calibration factor alpha(z) ---
fig, ax = plt.subplots()

ax.errorbar(
    cal["z"],
    cal["alpha"],
    yerr=cal["alpha_err"],
    fmt="o",
    capsize=2,
    color=c_midnight,
)

# snr > 8
ax.errorbar(
    cal_highsnr["z"],
    cal_highsnr["alpha"],
    yerr=cal_highsnr["alpha_err"],
    fmt="o",
    capsize=2,
    color=c_flatirons,
)

ax.set_xlabel("z")
ax.set_ylabel(r"$\alpha(z) = (dN/dX)_{\rm true} / (dN/dX)_{\rm meas}$")
ax.set_title("Effective completeness / calibration factor")
ax.grid(True, alpha=0.3)

plt.show()

## real data

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Apply mock-derived calibration factor alpha(z) to REAL dN/dX
# Assumes:
#   cal["z"], cal["alpha"], cal["alpha_err"] already computed from mocks
#   dndx68/dndx95 in file are absolute bounds [low, high]
# ============================================================

def bounds_to_asym_sigma(y, low, high):
    y = np.asarray(y, float)
    low = np.asarray(low, float)
    high = np.asarray(high, float)
    return (y - low, high - y)

def apply_alpha_to_dndx_bounds(
    z_cent, y, y68_low, y68_high, y95_low, y95_high,
    cal, *,
    include_alpha_uncertainty=True,
    clip_alpha=None,
):
    """
    Apply y_corr = alpha(z) * y to a measurement that stores absolute bounds.

    Propagate uncertainties (independent) with asymmetric lower/upper handled separately:
      frac_minus^2 = (sigma_minus/y)^2 + (sigma_alpha/alpha)^2
      frac_plus^2  = (sigma_plus /y)^2 + (sigma_alpha/alpha)^2
    """
    z_cent = np.asarray(z_cent, float)
    y = np.asarray(y, float)

    y68_low = np.asarray(y68_low, float)
    y68_high = np.asarray(y68_high, float)
    y95_low = np.asarray(y95_low, float)
    y95_high = np.asarray(y95_high, float)

    z_cal = np.asarray(cal["z"], float)
    alpha_cal = np.asarray(cal["alpha"], float)
    alpha_err_cal = np.asarray(cal["alpha_err"], float)

    alpha = np.interp(z_cent, z_cal, alpha_cal)
    alpha_err = np.interp(z_cent, z_cal, alpha_err_cal)

    if clip_alpha is not None:
        lo, hi = clip_alpha
        alpha = np.clip(alpha, lo, hi)

    y_corr = alpha * y

    # bounds -> asymmetric sigmas
    sig68_m, sig68_p = bounds_to_asym_sigma(y, y68_low, y68_high)
    sig95_m, sig95_p = bounds_to_asym_sigma(y, y95_low, y95_high)

    eps = 1e-30
    frac_alpha = alpha_err / np.maximum(alpha, eps) if include_alpha_uncertainty else 0.0

    frac68_m = sig68_m / np.maximum(y, eps)
    frac68_p = sig68_p / np.maximum(y, eps)
    frac95_m = sig95_m / np.maximum(y, eps)
    frac95_p = sig95_p / np.maximum(y, eps)

    sig68_m_corr = y_corr * np.sqrt(frac68_m**2 + frac_alpha**2)
    sig68_p_corr = y_corr * np.sqrt(frac68_p**2 + frac_alpha**2)
    sig95_m_corr = y_corr * np.sqrt(frac95_m**2 + frac_alpha**2)
    sig95_p_corr = y_corr * np.sqrt(frac95_p**2 + frac_alpha**2)

    y68_low_corr = y_corr - sig68_m_corr
    y68_high_corr = y_corr + sig68_p_corr
    y95_low_corr = y_corr - sig95_m_corr
    y95_high_corr = y_corr + sig95_p_corr

    return {
        "z_cent": z_cent,
        "alpha": alpha,
        "alpha_err": alpha_err,
        "y_raw": y,
        "y_corr": y_corr,
        "y68_low_corr": y68_low_corr,
        "y68_high_corr": y68_high_corr,
        "y95_low_corr": y95_low_corr,
        "y95_high_corr": y95_high_corr,
    }




In [ ]:
# ------------------------------------------------------------
# 1) Load REAL measurement from file
# ------------------------------------------------------------

subdir = "/Users/jibanmac/Documents/GitHub/desi_gpy_dla_detection_public/cddf_all/dla_cddf_20260107/desi_snr6.0_zqsos_2.15-5.0_dla_5.5_occam_1.0_dla_model_1_zminlyb_22.0_minobswave_3700.0_bins_5_lnhi_17.2-22.0_lnhi_dndx_17.2-19.0/"

# dN/dX: Fiducial
dndx_all = np.loadtxt(os.path.join(subdir, "dndx_all.txt"))
(_, N) = dndx_all.shape

dndx68 = np.full((N, 2), fill_value=np.nan)
dndx95 = np.full((N, 2), fill_value=np.nan)
(
    z_cent,
    dNdX,
    dndx68[:, 0],
    dndx68[:, 1],
    dndx95[:, 0],
    dndx95[:, 1],
) = dndx_all

# ------------- SNR > 8 -------------
subdir = "/Users/jibanmac/Documents/GitHub/desi_gpy_dla_detection_public/cddf_all/dla_cddf_20260107/desi_snr8.0_zqsos_2.15-5.0_dla_5.5_occam_1.0_dla_model_1_zminlyb_22.0_minobswave_3700.0_bins_5_lnhi_17.2-22.0_lnhi_dndx_17.2-19.0_z_dla_cddf_min_1.0_z_dla_dndx_min_2.0/"

# dN/dX: Fiducial
dndx_all = np.loadtxt(os.path.join(subdir, "dndx_all.txt"))
(_, N) = dndx_all.shape

dndx68_highsnr = np.full((N, 2), fill_value=np.nan)
dndx95_highsnr = np.full((N, 2), fill_value=np.nan)
(
    z_cent_highsnr,
    dNdX_highsnr,
    dndx68_highsnr[:, 0],
    dndx68_highsnr[:, 1],
    dndx95_highsnr[:, 0],
    dndx95_highsnr[:, 1],
) = dndx_all


# plot_line_density(z_cent, dNdX, dndx68, dndx95, color="blue", label="GP-LLS (SNR > 6)", bins_per_z=5, zmin=2, zmax=4.25)

# ------------------------------------------------------------
# 2) Apply calibration alpha(z) from mocks to REAL measurement
# ------------------------------------------------------------
# cal must already exist from your mock calibration step
corr_real = apply_alpha_to_dndx_bounds(
    z_cent=z_cent,
    y=dNdX,
    y68_low=dndx68[:, 0], y68_high=dndx68[:, 1],
    y95_low=dndx95[:, 0], y95_high=dndx95[:, 1],
    cal=cal,
    include_alpha_uncertainty=True,
    clip_alpha=None,
)

dndx68_corr = np.column_stack([corr_real["y68_low_corr"], corr_real["y68_high_corr"]])
dndx95_corr = np.column_stack([corr_real["y95_low_corr"], corr_real["y95_high_corr"]])

# ----------- SNR > 8 -----------
corr_real_highsnr = apply_alpha_to_dndx_bounds(
    z_cent=z_cent_highsnr,
    y=dNdX_highsnr,
    y68_low=dndx68_highsnr[:, 0], y68_high=dndx68_highsnr[:, 1],
    y95_low=dndx95_highsnr[:, 0], y95_high=dndx95_highsnr[:, 1],
    cal=cal_highsnr,
    include_alpha_uncertainty=True,
    clip_alpha=None,
)

dndx68_corr_highsnr = np.column_stack([corr_real_highsnr["y68_low_corr"], corr_real_highsnr["y68_high_corr"]])
dndx95_corr_highsnr = np.column_stack([corr_real_highsnr["y95_low_corr"], corr_real_highsnr["y95_high_corr"]])

# ------------------------------------------------------------
# 3) Plot: raw vs calibrated (bounds-style)
# ------------------------------------------------------------
fig, ax = plt.subplots()

plot_line_density(
    z_cent, dNdX, dndx68, dndx95,
    color="blue", label="DESI measured (raw) SNR $>$ 6",
    zmin=2, zmax=5,
    use_zcent_for_xerr=True,
)

plot_line_density(
    corr_real["z_cent"], corr_real["y_corr"], dndx68_corr, dndx95_corr,
    color=c_midnight, label="DESI measured (calibrated) SNR $>$ 6",
    zmin=2, zmax=5,
    use_zcent_for_xerr=True,
)

plot_line_density(
    corr_real_highsnr["z_cent"], corr_real_highsnr["y_corr"], dndx68_corr_highsnr, dndx95_corr_highsnr,
    color=c_flatirons, label="DESI measured (calibrated) SNR $>$ 8",
    zmin=2, zmax=5,
    use_zcent_for_xerr=True,
)

ax.set_xlabel("z")
ax.set_ylabel(r"$dN/dX$")
ax.set_title("LLS dN/dX: DESI raw vs calibrated (using mock alpha)")
ax.legend()
ax.grid(True, alpha=0.3)
# plt.show()

# # ------------------------------------------------------------
# # 4) Save calibrated file (same column format)
# # ------------------------------------------------------------
# out_arr = np.column_stack([
#     corr_real["z_cent"],
#     corr_real["y_corr"],
#     dndx68_corr[:, 0], dndx68_corr[:, 1],
#     dndx95_corr[:, 0], dndx95_corr[:, 1],
# ])

# np.savetxt(
#     os.path.join(subdir, "dndx_all_calibrated.txt"),
#     out_arr,
#     header="z_cent  dNdX_cal  dndx68_low  dndx68_high  dndx95_low  dndx95_high"
# )

# truth (from catalog)
plot_dndx(out_lls, label="Truth (from catalog)", prefer_boot=True, ax=ax, show=False)

plt.legend(fontsize=12)

## l(z) plot 


In [ ]:
import numpy as np

def dndx_to_ellz(z, dndx, omega_m=0.279):
    """
    Convert dN/dX -> ell(z)=dN/dz using:
        dX/dz = (1+z)^2 / E(z),
        E(z) = sqrt(Ωm(1+z)^3 + (1-Ωm))
    """
    z = np.asarray(z, dtype=float)
    dndx = np.asarray(dndx, dtype=float)

    Ez = np.sqrt(omega_m * (1.0 + z)**3 + (1.0 - omega_m))
    dX_dz = (1.0 + z)**2 / Ez
    return dndx * dX_dz

def dndx_bounds_to_ellz(z, bounds, omega_m=0.279):
    """
    Convert absolute bounds [low, high] on dN/dX into bounds on dN/dz.
    """
    z = np.asarray(z, float)
    bounds = np.asarray(bounds, float)

    Ez = np.sqrt(omega_m * (1.0 + z)**3 + (1.0 - omega_m))
    dX_dz = (1.0 + z)**2 / Ez

    ell_low = bounds[:, 0] * dX_dz
    ell_high = bounds[:, 1] * dX_dz
    return np.column_stack([ell_low, ell_high])

In [ ]:
# --- raw ---
ell_raw = dndx_to_ellz(z_cent, dNdX, omega_m=0.279)
ell68_raw = dndx_bounds_to_ellz(z_cent, dndx68, omega_m=0.279)
ell95_raw = dndx_bounds_to_ellz(z_cent, dndx95, omega_m=0.279)

# --- calibrated ---
ell_corr = dndx_to_ellz(
    corr_real["z_cent"],
    corr_real["y_corr"],
    omega_m=0.279
)
ell68_corr = dndx_bounds_to_ellz(
    corr_real["z_cent"],
    np.column_stack([corr_real["y68_low_corr"], corr_real["y68_high_corr"]]),
    omega_m=0.279
)
ell95_corr = dndx_bounds_to_ellz(
    corr_real["z_cent"],
    np.column_stack([corr_real["y95_low_corr"], corr_real["y95_high_corr"]]),
    omega_m=0.279
)

# --- SNR > 8 raw ---
ell_raw_highsnr = dndx_to_ellz(z_cent_highsnr, dNdX_highsnr, omega_m=0.279)
ell68_raw_highsnr = dndx_bounds_to_ellz(z_cent_highsnr, dndx68_highsnr, omega_m=0.279)
ell95_raw_highsnr = dndx_bounds_to_ellz(z_cent_highsnr, dndx95_highsnr, omega_m=0.279)

# --- SNR > 8 calibrated ---
ell_corr_highsnr = dndx_to_ellz(
    corr_real_highsnr["z_cent"],
    corr_real_highsnr["y_corr"],
    omega_m=0.279
)
ell68_corr_highsnr = dndx_bounds_to_ellz(
    corr_real_highsnr["z_cent"],
    np.column_stack([corr_real_highsnr["y68_low_corr"], corr_real_highsnr["y68_high_corr"]]),
    omega_m=0.279
)
ell95_corr_highsnr = dndx_bounds_to_ellz(
    corr_real_highsnr["z_cent"],
    np.column_stack([corr_real_highsnr["y95_low_corr"], corr_real_highsnr["y95_high_corr"]]),
    omega_m=0.279
)



In [ ]:
# #0 Prochaska+2010; 1 Fumagalli+2013; 2 Crighton+2018; 3 O'Meara+2013; 3# Ribaudo+2011; 4 BOSS/This work
# zleft zright lz lzuperr lzdwerr ref

literature_data = np.loadtxt("../../DLA_data/lofz_literature.txt")

# read columns
zleft = literature_data[:, 0]
zright = literature_data[:, 1]
lz = literature_data[:, 2]
lzuperr = literature_data[:, 3]
lzdwerr = literature_data[:, 4]
ref = literature_data[:, 5].astype(int)

ref_dict = {
    0: "SDSS: Prochaska+2010",
    1: "MagE: Fumagalli+2013",
    2: "GGG: Crighton+2018",
    3: "HST: O'Meara+2013",
    # 4: "Ribaudo+2011",
    4: "BOSS: Fumagalli+2020",
}


# make plot of literature values
fig, ax = plt.subplots()    

for iref in range(5):
    m = ref == iref
    if np.any(m):
        zcen_lit = 0.5 * (zleft[m] + zright[m])
        ax.errorbar(
            zcen_lit,
            lz[m],
            yerr=[lzdwerr[m], lzuperr[m]],
            fmt="o",
            label=ref_dict[iref],
            capsize=2,
        )
plt.legend()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()


# --- raw ---
ax.errorbar(
    z_cent,
    ell_raw, 
    yerr=[(ell_raw - ell68_raw[:, 0]), (ell68_raw[:, 1] - ell_raw)],
    color=c_skyline, lw=2, label="DESI GP (raw)", marker="o"
)
ax.fill_between(
    z_cent,
    ell95_raw[:, 0], ell95_raw[:, 1],
    color=c_skyline, alpha=0.15
)

# --- calibrated ---
ax.errorbar(
    corr_real["z_cent"],
    ell_corr,
    yerr=[(ell_corr - ell68_corr[:, 0]), (ell68_corr[:, 1] - ell_corr)],
    color=c_flatirons, lw=2, label="DESI GP (calibrated)", marker="o"
)
ax.fill_between(
    corr_real["z_cent"],
    ell95_corr[:, 0], ell95_corr[:, 1],
    color=c_flatirons, alpha=0.15
)

# --- SNR > 8 calibrated ---
ax.errorbar(
    corr_real_highsnr["z_cent"],
    ell_corr_highsnr,
    yerr=[(ell_corr_highsnr - ell68_corr_highsnr[:, 0]), (ell68_corr_highsnr[:, 1] - ell_corr_highsnr)],
    color=c_flatirons_ll, lw=2, label="DESI GP (calibrated, SNR $>$ 8)", marker="o"
)
ax.fill_between(
    corr_real_highsnr["z_cent"],
    ell95_corr_highsnr[:, 0], ell95_corr_highsnr[:, 1],
    color=c_flatirons_ll, alpha=0.15
)

# literature values
# from Michelle Fumagalli's lofz_literature.txt
for iref in range(5):
    m = ref == iref
    if np.any(m):
        zcen_lit = 0.5 * (zleft[m] + zright[m])
        ax.errorbar(
            zcen_lit,
            lz[m],
            yerr=[lzdwerr[m], lzuperr[m]],
            fmt="o",
            label=ref_dict[iref],
            capsize=2,
        )


ax.set_xlabel("z")
ax.set_ylabel(r"$\ell(z) \equiv dN/dz$")
ax.set_title("LLS line density from GP-LLS DESI vs Literature")
ax.legend(fontsize=14)
ax.grid(True, alpha=0.3)

plt.show()